# ⚽ T-DEED: Ball Action Spotting (Pass & Shot Detection)
### End-to-End Training & Fine-Tuning Pipeline on Kaggle (SoccerNet SN-BAS-2024)

**Objective**: Replace error-prone kinematic ball-tracking heuristics with a temporal deep-learning model (**T-DEED**) to spot football events (**Passes, Shots, Goals, Crosses, Headers, Tackles**) directly from broadcast video frames.

**Why T-DEED?**:
- **Winner of SoccerNet Ball Action Spotting 2024** & baseline for 2025.
- **Robust to Ball Occlusions**: Recognizes player body language, leg swing, and collective movement rather than relying on noisy 2D ball bounding boxes.
- **12 Football Event Classes**: `Pass`, `Drive`, `Header`, `High Pass`, `Out`, `Cross`, `Throw In`, `Shot`, `Ball-Player Block`, `Player Successful Tackle`, `Free Kick`, `Goal`.
- **Lightweight Architecture**: RegNetY-200MF (`rny002_gsf`) backbone + Temporal SGP Mixer (~49 MB checkpoint, runs efficiently on Kaggle T4 GPU).

--- 
## Step 1: Install Dependencies (Run BEFORE any Python imports)
Installing all upgraded packages first prevents in-memory kernel conflicts with pre-installed Kaggle packages.

In [15]:
# Step 1: Upgrade huggingface_hub and SoccerNet before importing anything
!pip install -q -U "huggingface_hub>=0.23.0" SoccerNet
!pip install -q einops moviepy av scikit-learn wandb
!nvidia-smi

Sat Oct  3 04:40:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

--- 
## Step 2: Hugging Face Authentication (No Password Required)
SoccerNet now authenticates via Hugging Face access tokens (no deprecated FTP password needed).
Add your token to Kaggle **Add-ons -> Secrets** under the key `HF_TOKEN`, or enter it directly when prompted.

In [16]:
import os

# Check for Kaggle Secret first
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ Found HF_TOKEN in Kaggle Secrets.")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")

if not hf_token:
    import getpass
    hf_token = getpass.getpass("Enter your Hugging Face Access Token: ")

os.environ["HF_TOKEN"] = hf_token
# Authenticate CLI and environment
!huggingface-cli login --token $hf_token
print("✅ Successfully authenticated with Hugging Face.")

✅ Found HF_TOKEN in Kaggle Secrets.
/bin/bash: line 1: huggingface-cli: command not found
✅ Successfully authenticated with Hugging Face.


--- 
## Step 3: Clone T-DEED Repository & Configure Environment
Set up the T-DEED workspace and disable interactive wandb prompts.

In [17]:
import os

# Set working directory
WORKSPACE_DIR = "/kaggle/working"
TDEED_DIR = os.path.join(WORKSPACE_DIR, "T-DEED")

# Clone repository if not present
if not os.path.exists(TDEED_DIR):
    !git clone https://github.com/arturxe2/T-DEED.git {TDEED_DIR}

%cd {TDEED_DIR}

# Disable wandb interactive prompt
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"

/kaggle/working/T-DEED


--- 
## Step 4: Download SoccerNet Ball Action Spotting 2024 Dataset
We download videos to `/kaggle/temp/SoccerNet` to preserve Kaggle's `/kaggle/working` 20GB disk limit.

Executed in an isolated subprocess to ensure clean, conflict-free package loading.

In [18]:
import os

DATA_DIR = "/kaggle/temp/SoccerNet"
os.makedirs(DATA_DIR, exist_ok=True)

print("📥 Downloading SoccerNet Ball Action Spotting 2024 annotations and videos...")
# Run in clean subprocess (completely avoids any Jupyter in-memory module caching)
!python -c "from SoccerNet.Downloader import SoccerNetDownloader as SNdl; d = SNdl(LocalDirectory='/kaggle/temp/SoccerNet'); d.downloadDataTask(task='spotting-ball-2024', split=['valid'])"

# Fallback if needed
if not any(os.scandir(DATA_DIR)):
    print("Notice: Running direct huggingface-cli download...")
    !huggingface-cli download SoccerNet/SN-BAS-2024 --repo-type dataset --local-dir /kaggle/temp/SoccerNet

# Uncomment the following line when ready for full training set download:
# !python -c "from SoccerNet.Downloader import SoccerNetDownloader as SNdl; d = SNdl(LocalDirectory='/kaggle/temp/SoccerNet'); d.downloadDataTask(task='spotting-ball-2024', split=['train'])"

print("✅ Download complete. Directory contents:")
!ls -lh {DATA_DIR}

📥 Downloading SoccerNet Ball Action Spotting 2024 annotations and videos...
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 1 files:   0%|                                   | 0/1 [00:00<?, ?it/s]
Reconstructing (incomplete total...):  89%|███████ | 1.81GB / 2.04GB,  391MB/s  
Reconstructing (incomplete total...): 100%|████████| 2.04GB / 2.04GB,  485MB/s  

Fetching 1 files: 100%|███████████████████████████| 1/1 [00:05<00:00,  5.98s/it]
Download complete: █████████████████████████████████████████| 2.04GB,  403MB/s  
Reconstruction complete: 100%|█████████████████████| 2.04GB / 2.04GB,  601MB/s  
Download complete: █████████████████████████████████████████| 2.04GB,  403MB/s  
✅ Download complete. Directory contents:
total 4.0K
drwxr-xr-x 3 root root 4.0K Oct  3 04:40 spotting-ball-2024


--- 
## Step 5: Extract Frames at Model Input Resolution (796 × 448)
T-DEED processes extracted 25 fps JPEG frames scaled to $796 \times 448$.
We run the extraction script utilizing multi-processing workers into `/kaggle/temp/frames_snb`.

In [19]:
import os

FRAMES_DIR = "/kaggle/temp/frames_snb"
os.makedirs(FRAMES_DIR, exist_ok=True)

print("🚀 Extracting frames from 720p videos to:", FRAMES_DIR)
!python extract_frames_snb.py \
    --video_dir {DATA_DIR} \
    -o {FRAMES_DIR} \
    --sample_fps 25 \
    --num_workers 4

print("✅ Frames extraction finished.")

🚀 Extracting frames from 720p videos to: /kaggle/temp/frames_snb
/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
error: XDG_RUNTIME_DIR is invalid or not set in the environment.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5208:(_snd_config_ev

--- 
## Step 6: Configure T-DEED for Kaggle
We set up:
1. `data/soccernetball/labels_path.txt`: points to the downloaded annotation directory.
2. `config/SoccerNetBall/SoccerNetBall_kaggle.json`: specifies batch size, paths, learning rate, and backbone (`rny002_gsf`).

In [20]:
import json
import os

# 1. Update label path pointer
LABELS_FILE = os.path.join(TDEED_DIR, "data", "soccernetball", "labels_path.txt")
os.makedirs(os.path.dirname(LABELS_FILE), exist_ok=True)
with open(LABELS_FILE, "w") as f:
    f.write(f"{DATA_DIR}\n")

print(f"✅ Updated labels path pointer: {DATA_DIR}")

# 2. Create optimized Kaggle configuration
SAVE_DIR = "/kaggle/working/checkpoints/SoccerNetBall_kaggle"
STORE_DIR = "/kaggle/temp/stored_partitions"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(STORE_DIR, exist_ok=True)

kaggle_config = {
    "frame_dir": FRAMES_DIR,
    "save_dir": SAVE_DIR,
    "store_dir": STORE_DIR,
    "store_mode": "store",  # First run generates partitions, then switches to 'load'
    "batch_size": 4,
    "clip_len": 100,
    "crop_dim": -1,
    "dataset": "soccernetball",
    "radi_displacement": 4,
    "epoch_num_frames": 200000,  # Optimized for fast Kaggle epochs
    "feature_arch": "rny002_gsf",  # RegNetY-200MF (lightweight, fits on T4)
    "learning_rate": 0.0008,
    "mixup": True,
    "modality": "rgb",
    "num_classes": 12,
    "num_epochs": 25,
    "warm_up_epochs": 3,
    "start_val_epoch": 5,
    "temporal_arch": "ed_sgp_mixer",
    "n_layers": 2,
    "sgp_ks": 9,
    "sgp_r": 4,
    "only_test": False,
    "criterion": "map",
    "num_workers": 2
}

CONFIG_PATH = os.path.join(TDEED_DIR, "config", "SoccerNetBall", "SoccerNetBall_kaggle.json")
with open(CONFIG_PATH, "w") as f:
    json.dump(kaggle_config, f, indent=4)

print(f"✅ Generated configuration: {CONFIG_PATH}")

✅ Updated labels path pointer: /kaggle/temp/SoccerNet
✅ Generated configuration: /kaggle/working/T-DEED/config/SoccerNetBall/SoccerNetBall_kaggle.json


--- 
## Step 7: Pre-Flight Check: Generate Partitions (`store_mode: store`)
T-DEED requires generating dataset clip index partitions on the first run. Once created, the script exits cleanly, after which we switch to `load` mode.

In [21]:
# Run partition generation in store mode
!python train_tdeed.py --model SoccerNetBall_kaggle

# Switch config to 'load' mode for continuous training
with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)
cfg["store_mode"] = "load"
with open(CONFIG_PATH, "w") as f:
    json.dump(cfg, f, indent=4)

print("✅ Partitions cached successfully. Config switched to store_mode='load'.")

Setting seed to:  1
Dataset size: 2000
  0%|                                                     | 0/4 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/kaggle/working/T-DEED/train_tdeed.py", line 321, in <module>
    main(get_args())
    ~~~~^^^^^^^^^^^^
  File "/kaggle/working/T-DEED/train_tdeed.py", line 118, in main
    classes, pretrain_classes, train_data, val_data, val_data_frames = get_datasets(args)
                                                                       ~~~~~~~~~~~~^^^^^^
  File "/kaggle/working/T-DEED/dataset/datasets.py", line 38, in get_datasets
    train_data = ActionSpotDataset(
        classes, os.path.join('data', args.dataset, 'train.json'),
        args.frame_dir, args.store_dir, args.store_mode,
        args.modality, args.clip_len, dataset_len, **dataset_kwargs)
  File "/kaggle/working/T-DEED/dataset/frame.py", line 93, in __init__
    self._store_clips()
    ~~~~~~~~~~~~~~~~~^^
  File "/kaggle/working/T-DEED/dataset/frame.py", line 114, i

--- 
## Step 8: Train Model
Train T-DEED using the Linear Warmup + Cosine Annealing learning rate schedule.
Checkpoints are saved automatically to `/kaggle/working/checkpoints/SoccerNetBall_kaggle/`.

In [22]:
!python train_tdeed.py --model SoccerNetBall_kaggle

Setting seed to:  1
Dataset size: 2000
Traceback (most recent call last):
  File "/kaggle/working/T-DEED/train_tdeed.py", line 321, in <module>
    main(get_args())
    ~~~~^^^^^^^^^^^^
  File "/kaggle/working/T-DEED/train_tdeed.py", line 118, in main
    classes, pretrain_classes, train_data, val_data, val_data_frames = get_datasets(args)
                                                                       ~~~~~~~~~~~~^^^^^^
  File "/kaggle/working/T-DEED/dataset/datasets.py", line 38, in get_datasets
    train_data = ActionSpotDataset(
        classes, os.path.join('data', args.dataset, 'train.json'),
        args.frame_dir, args.store_dir, args.store_mode,
        args.modality, args.clip_len, dataset_len, **dataset_kwargs)
  File "/kaggle/working/T-DEED/dataset/frame.py", line 95, in __init__
    self._load_clips()
    ~~~~~~~~~~~~~~~~^^
  File "/kaggle/working/T-DEED/dataset/frame.py", line 202, in _load_clips
    with open(store_path + '/frame_paths.pkl', 'rb') as f:
         ~

--- 
## Step 9: Evaluate on Validation Split (mAP Calculation)
Run test evaluation across tight temporal tolerances (delta = 1s, 2s) to measure spotting precision for each of the 12 classes.

In [23]:
# Set only_test = True in config
with open(CONFIG_PATH, "r") as f:
    cfg = json.load(f)
cfg["only_test"] = True
with open(CONFIG_PATH, "w") as f:
    json.dump(cfg, f, indent=4)

!python train_tdeed.py --model SoccerNetBall_kaggle

Setting seed to:  1
Dataset size: 2000
Traceback (most recent call last):
  File "/kaggle/working/T-DEED/train_tdeed.py", line 321, in <module>
    main(get_args())
    ~~~~^^^^^^^^^^^^
  File "/kaggle/working/T-DEED/train_tdeed.py", line 118, in main
    classes, pretrain_classes, train_data, val_data, val_data_frames = get_datasets(args)
                                                                       ~~~~~~~~~~~~^^^^^^
  File "/kaggle/working/T-DEED/dataset/datasets.py", line 38, in get_datasets
    train_data = ActionSpotDataset(
        classes, os.path.join('data', args.dataset, 'train.json'),
        args.frame_dir, args.store_dir, args.store_mode,
        args.modality, args.clip_len, dataset_len, **dataset_kwargs)
  File "/kaggle/working/T-DEED/dataset/frame.py", line 95, in __init__
    self._load_clips()
    ~~~~~~~~~~~~~~~~^^
  File "/kaggle/working/T-DEED/dataset/frame.py", line 202, in _load_clips
    with open(store_path + '/frame_paths.pkl', 'rb') as f:
         ~

--- 
## Step 10: Run Inference Test on Video & Package Weights
Verify event spotting predictions on a video clip and package the best checkpoint for download.

In [25]:
import glob
import shutil

# Locate best checkpoint
checkpoints = glob.glob(os.path.join(SAVE_DIR, "*.pt"))
print("Available checkpoints:", checkpoints)

# Package checkpoint and config into export directory
EXPORT_DIR = "/kaggle/working/tdeed_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

for cp in checkpoints:
    shutil.copy(cp, EXPORT_DIR)
shutil.copy(CONFIG_PATH, EXPORT_DIR)

!tar -czvf /kaggle/working/tdeed_soccernetball_best.tar.gz -C {EXPORT_DIR} .
print("\n🎉 Model weights packaged to: /kaggle/working/tdeed_soccernetball_best.tar.gz")
print("You can now download this archive directly from Kaggle's Output panel!")

Available checkpoints: []
./
./SoccerNetBall_kaggle.json

🎉 Model weights packaged to: /kaggle/working/tdeed_soccernetball_best.tar.gz
You can now download this archive directly from Kaggle's Output panel!
